# 🎒 Schulbedarf NLP Product Matcher Training Pipeline

Dieses Notebook dient zum Trainieren und Evaluieren des Klassifikationsmodells, das Textzeilen aus Schulmateriallisten direkt auf konkrete Produkte im Katalog (`products.csv`) abbildet.

In [ ]:
import pandas as pd
import pickle
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

## 1. Mappings & Katalog-Daten laden
Wir laden den durch den lokalen NLP-Matcher erzeugten Mappings-Datensatz.

In [ ]:
mapped_path = "data/pdf_lines_mapped.csv"
products_path = "data/products.csv"

if not os.path.exists(mapped_path):
    raise FileNotFoundError(f"Mappings-Datei nicht gefunden unter: {mapped_path}")

df_mapped = pd.read_csv(mapped_path)
print(f"Erfolgreich geladen: {len(df_mapped)} Einträge.")
df_mapped.head(10)

## 2. Trainingsdaten vorbereiten
Wir entfernen leere Zeilen und teilen den Datensatz in Features (X) und Klassen-Labels (y).

In [ ]:
df_mapped['raw_line'] = df_mapped['raw_line'].fillna("")
df_mapped = df_mapped[df_mapped['raw_line'].str.strip() != ""]

X = df_mapped['raw_line'].values
y = df_mapped['product_id'].values

print(f"Training mit {len(X)} Zeilen für {len(set(y))} eindeutige Produkt-IDs.")

## 3. Pipeline trainieren
Wir verwenden einen TF-IDF Vektorisierer auf Zeichen-N-Gramm Basis, um robust gegenüber OCR-Fehlern zu sein, kombiniert mit einer logistischen Regression.

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 6),
    min_df=1,
    sublinear_tf=True
)

classifier = LogisticRegression(
    C=20.0,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

pipeline = make_pipeline(vectorizer, classifier)
pipeline.fit(X, y)

## 4. Evaluierung

In [ ]:
train_acc = pipeline.score(X, y)
print(f"Klassifikationsgenauigkeit auf Trainingsdaten: {train_acc * 100:.2f}%")

## 5. Modell als PKL exportieren

In [ ]:
model_path = "src/model.pkl"
model_data = {
    "pipeline": pipeline,
    "classes": classifier.classes_
}

with open(model_path, "wb") as f:
    pickle.dump(model_data, f)

print(f"Modell erfolgreich gespeichert unter: {model_path}")